# SALoRA — Step 4 training matrix on Colab GPU

Runs the same 10-condition initial pass (5 methods x ANLI R1+R2 x seed 42) that was
running on a MacBook Air's MPS backend, on a real CUDA GPU instead. Same code, same
data, same hyperparameters, same allocations -- only the hardware changes.

**Before running, do these two things:**

1. **Runtime > Change runtime type > T4 GPU** (or better, if you have Colab Pro).
2. **Upload the source model checkpoint to Google Drive.** The repo's `models/` folder
   is git-ignored (460MB+ checkpoint, never committed), so it has to come from
   somewhere else. Zip your local `models/source_roberta/` folder into
   `source_roberta.zip` and upload it to your Google Drive at
   `MyDrive/salora/source_roberta.zip` (change the path in the cell below if you put
   it elsewhere).

If `github.com/bindhupagadala10/salora` is a **private** repo, also add a GitHub
Personal Access Token as a Colab secret named `GH_TOKEN` (key icon in the left
sidebar > "Secrets" > Add new secret) before running the clone cell. If the repo is
public, you can skip this -- the notebook handles both cases automatically.

## 1. Mount Google Drive (for the checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo
Uses a `GH_TOKEN` Colab secret if you set one (needed only for a private repo).

In [ ]:
import subprocess

try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None

REPO_URL = "https://github.com/bindhupagadala10/salora.git"
if GH_TOKEN:
    REPO_URL = f"https://{GH_TOKEN}@github.com/bindhupagadala10/salora.git"

!git clone -b benchmark-selection {REPO_URL} 2>&1 | tail -20
%cd salora
!git log -1 --oneline
!git log -1 --format='%cd' --date=relative

**Sanity check:** the line above should show commit `1041069` ("Wire literature
review into the actual paper draft...") or later. If it shows an older commit, the
local Mac repo hasn't been pushed yet -- go run `git push origin benchmark-selection`
there first, then re-run the clone cell above.

## 3. Install dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type > GPU")

## 4. Unpack the source model checkpoint from Drive

In [ ]:
import os

DRIVE_ZIP = "/content/drive/MyDrive/salora/source_roberta.zip"  # change if you put it elsewhere
assert os.path.exists(DRIVE_ZIP), (
    f"Not found: {DRIVE_ZIP}\n"
    "Upload source_roberta.zip to your Google Drive first (see the intro cell), "
    "or edit DRIVE_ZIP above to point at wherever you put it."
)

!unzip -oq "{DRIVE_ZIP}" -d .
!ls -la models/source_roberta

## 5. Verify the PEFT mechanism before spending GPU time
Same check that was run locally (Step 3) -- confirms per-layer ranks apply correctly
and the parameter counts match, on this environment's peft/transformers versions.

In [ ]:
!python -m src.salora.verify_peft_mechanism

**Check before continuing:** all 4 checks should read the same as they did locally --
Check 1 PASS, Check 2 "apparent FAIL" (explained by Check 4), Check 3 PASS, Check 4
FAIL (classifier head auto-trainable, 592,899 params, constant across methods). If
anything differs from that pattern here, stop and figure out why before training --
it would mean the parameter-budget-equality assumption doesn't hold on this
environment's peft version.

## 6. Run the training matrix

5 methods x 2 targets (ANLI R1, ANLI R2) x seed 42 = 10 runs. One combination
(`anli_r1` / `salora_mmd` / seed 42) already completed locally before the Mac job was
stopped in favor of this GPU run -- it's included again here for a clean, fully
GPU-consistent set of 10, but feel free to comment it out of `COMBOS` below if you'd
rather not repeat it (it's a ~2-minute rerun on GPU either way, so not worth the
bookkeeping to skip it).

In [ ]:
import subprocess, os, time

TARGETS = ["anli_r1", "anli_r2"]
METHODS = ["salora_mmd", "salora_sinkhorn", "random", "inverse_mmd", "inverse_sinkhorn"]
SEED = 42

COMBOS = [(t, m) for t in TARGETS for m in METHODS]

os.makedirs("logs/salora_step4_colab", exist_ok=True)

results_summary = []
for target, method in COMBOS:
    logfile = f"logs/salora_step4_colab/{target}_{method}_seed{SEED}.log"
    print(f"=== starting {target} / {method} / seed{SEED} ===")
    t0 = time.time()
    with open(logfile, "w") as f:
        proc = subprocess.run(
            ["python", "-u", "-m", "src.salora.train_salora",
             "--dataset", target, "--method", method, "--seed", str(SEED)],
            stdout=f, stderr=subprocess.STDOUT,
        )
    elapsed = time.time() - t0
    status = "OK" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
    print(f"=== {status}: {target} / {method} -- {elapsed/60:.1f} min -- see {logfile} ===")
    results_summary.append((target, method, status, elapsed))

print("\n=== ALL DONE ===")
for target, method, status, elapsed in results_summary:
    print(f"{target:10s} {method:18s} {status:20s} {elapsed/60:.1f} min")

## 7. Review results

In [ ]:
import pandas as pd

df = pd.read_csv("results/experiments.csv")
matrix = df[df["Experiment"] == "SALoRA Matrix"]
matrix[["Target", "Method", "Metric", "Seed", "Total_Trainable_Params", "Accuracy", "Macro F1"]]

## 8. Persist results back to GitHub (manual, run only when ready)

This is deliberately **not** wired to auto-run with the cells above. Check the diff
first, then commit and push explicitly when you're satisfied with the results.

In [ ]:
# Step 1: look before you commit.
!git status
!git diff --stat results/experiments.csv

In [ ]:
# Step 2: only run this once you've reviewed the diff above and are ready to push.
# git needs an identity in this fresh Colab environment -- set yours here first.
!git config user.email "you@example.com"
!git config user.name "Your Name"

!git add results/experiments.csv
!git commit -m "Colab GPU run: SALoRA step 4 initial-pass matrix (ANLI R1+R2, 5 methods, seed 42)"

REPO_URL_PUSH = f"https://{GH_TOKEN}@github.com/bindhupagadala10/salora.git" if GH_TOKEN else None
if REPO_URL_PUSH:
    !git push {REPO_URL_PUSH} benchmark-selection
else:
    !git push origin benchmark-selection